<a href="https://colab.research.google.com/github/claramanolache/ML_Intro/blob/main/Week_6_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6: Practice Notebook

**You do not need to submit this Notebook.** It is for practice only.

## Introduction to the Dataset

The Penguins dataset was collected by the Palmer Station, Antarctica Long-Term Ecological Research program. It comprises measurements from three penguin species - Adélie, Gentoo, and Chinstrap - living on islands in the Palmer Archipelago, Antarctica. The dataset includes numerical features such as bill length and depth, flipper length, and body mass, as well as categorical features like species, island, and sex.

### Goal
Explore and apply Decision Trees for both classification and regression using the Penguins dataset.

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import compose
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import tree

## Load the Dataset

This dataset is available from the Seaborn library (usually abbreviated as `sns`).

In [ ]:
df = sns.load_dataset("penguins")
df.shape

### Inspect the Data Types

Most of the columns are real numbers (measurements of the penguins), with three categorical columns. We can also see that there are a few null values, so we should remove these rows.

In [ ]:
df.info()

## Preparing the Data

The following variable will be used to set a random seed for certain processes. Using a seed makes random processes deterministic, meaning this notebook will do things like split data the same way on each run.

You are free to change this value to explore how the notebook runs differently.

In [ ]:
seed = 0

### Removing Null Values

We have seen previously how to remove null values after splitting a dataset, which better reflects many production scenarios. However, because this is a small dataset intended for light experimentation, for simplicity we will remove null values as an initial step.

In [ ]:
df.isnull().sum()
df = df.dropna()
df.shape

### Creating a Feature Matrix and Target Vector

For classificaion, we will use penguin species as the label.

In [ ]:
X = df.drop(columns="species")
y = df["species"]
X.shape, y.shape

### Splitting the Data into Training and Test Sets

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=seed
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
X_train.head()

### Encoding Categorical Features

In order to use categorical features with a Decision Tree, we need to encode them. Since none of these features have any ordinal relationship to one another, we will use one-hot encoding.

Decision Trees assess splits independently on each feature. Thus, it is not necessary to scale features. Below, we simply "pass" numerical features through the transformer; this step ensures that the output of the transformer still contains these unmodified features.

In [ ]:
transformer = compose.make_column_transformer(
    (
        # Encode non-numerical features.
        preprocessing.OneHotEncoder(),
        compose.make_column_selector(dtype_include=object)
    ),
    (
        # Pass numerical features through without transforming.
        "passthrough",
        compose.make_column_selector(dtype_include=np.number)
    ),
    verbose_feature_names_out=False
)
X_train_array = transformer.fit_transform(X_train)
X_test_array = transformer.transform(X_test)

Transformers output NumPy arrays. In order to continue using DataFrames (which provide many convenience methods), we can pass these arrays into a DataFrame constructor using the column names that the transformer created.

In [ ]:
columns = transformer.get_feature_names_out()
X_train = pd.DataFrame(X_train_array, columns=columns)
X_test = pd.DataFrame(X_test_array, columns=columns)
X_train.shape, X_test.shape

In [ ]:
X_train.head()

## Visualizing the Dataset

Let's visualize the relationship between some features to understand the data better. Below, we use bill length and flipper length as the features, with different penguin species represented by different colors.

In [ ]:
X_y_train = pd.merge(
    X_train,
    y_train.reset_index(drop=True),
    left_index=True,
    right_index=True
)
X_y_train.shape

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=X_y_train,
    x="bill_length_mm",
    y="flipper_length_mm",
    hue="species",
    s=100
)
plt.title("Penguins: Bill Length vs Flipper Length")
plt.xlabel("Bill Length (mm)")
plt.ylabel("Flipper Length (mm)")
plt.legend(title="Species")
plt.show()

## Classification Task: Predicting Penguin Species

We will build a Decision Tree Classifier to predict the species of a penguin based on its physical measurements.

### Understanding Regularization Parameters

- **max_depth**: Limits the maximum depth of the tree. Lower values prevent overfitting by creating shallower trees.
- **min_samples_split**: The minimum number of samples required to split a node. Higher values prevent the tree from creating splits based on very few samples.
- **min_samples_leaf**: The minimum number of samples required at a leaf node. Higher values smooth the model and prevent overfitting.
- **max_leaf_nodes**: Limits the maximum number of leaf nodes. This controls the overall complexity of the tree.
- **max_features**: The number of features to consider when looking for the best split. Lower values add randomness and can reduce overfitting.

### Train a Decision Tree Classifier

We will train a Decision Tree Classifier using the Gini index as the splitting criterion.

In [ ]:
clf = tree.DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    max_leaf_nodes=None,
    max_features=None,
    random_state=seed
)

clf.fit(X_train, y_train)

### Visualize the Decision Tree

Visualizing the tree helps us understand how the model makes decisions at each split. The tree shows which features are used for splitting and the Gini impurity at each node.

For example, `flipper_length_mm <= 206.5` means that if a test sample's value for this feature is less than 206.5, the predictor will traverse to the right (True) side of the tree when looking for a leaf; otherwise, it will traverse to the left (False) side of the tree.

Once the predictor reaches a leaf, that class will be its prediction for the sample.

In [ ]:
plt.figure(figsize=(20, 10))
tree.plot_tree(
    clf,
    feature_names=X_train.columns,
    class_names=clf.classes_,
    filled=True,
    fontsize=10
)
plt.title("Decision Tree Classifier for Penguin Species")
plt.show()

### Evaluate the Classifier

Let's evaluate the model's performance on the test set.

In [ ]:
y_pred = clf.predict(X_test)
metrics.accuracy_score(y_test, y_pred)

In [ ]:
cm = metrics.confusion_matrix(y_test, y_pred)
disp = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=clf.classes_
)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix for Penguin Species Classification")
plt.show()

## Regression Task: Predicting Body Mass

Now let's use a Decision Tree Regressor to predict the body mass of penguins based on their other physical measurements.

### Train a Decision Tree Regressor

For regression, the Decision Tree minimizes the variance (or mean squared error) at each split instead of using Gini impurity. You can experiment with the same regularization parameters.

In [ ]:
X_train.head()

We can create features for a regression problem by selecting a few real-valued features, with one - body mass in this case - used as the target vector.

In [ ]:
X_train_reg = X_train[["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]]
y_train_reg = X_train["body_mass_g"]
X_test_reg = X_test[["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]]
y_test_reg = X_test["body_mass_g"]

X_train_reg.shape, y_train_reg.shape, X_test_reg.shape, y_test_reg.shape

In [ ]:
reg = tree.DecisionTreeRegressor(
    max_depth=3,
    min_samples_split=5,
    min_samples_leaf=2,
    max_leaf_nodes=None,
    max_features=None,
    random_state=seed
)

reg.fit(X_train_reg, y_train_reg)

### Visualize the Regression Tree

For regression trees, each leaf node contains the predicted value - average body mass in this case - instead of a class label. This means that, upon reaching a leaf node, Decision Tree regressors only can predict an average based on the samples the model observed and split on in the training set.

In other words, two test samples will result in the same prediction if the predictor reaches the same leaf when traversing the tree.

In [ ]:
plt.figure(figsize=(20, 10))
tree.plot_tree(
    reg,
    feature_names=X_train_reg.columns,
    filled=True,
    fontsize=10
)
plt.title("Decision Tree Regressor for Penguin Body Mass")
plt.show()

### Evaluate the Regressor

Let's evaluate the regression model using the Root Mean Squared Error (RMSE).

In [ ]:
y_pred_reg = reg.predict(X_test_reg)
rmse = metrics.root_mean_squared_error(y_test_reg, y_pred_reg)
rmse

Finally, we can plot the actual versus predicted body mass values. The red line represents what a perfect predictive output.

Notice how the predictions appear at set values with respect to the y-axis. This happens because the Decision Tree regressor can only predict a fixed number of different values: one for each leaf in the tree.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)
plt.plot(
    [y_test_reg.min(), y_test_reg.max()],
    [y_test_reg.min(), y_test_reg.max()],
    "r--",
    lw=2
)
plt.xlabel("Actual Body Mass (g)")
plt.ylabel("Predicted Body Mass (g)")
plt.title("Actual vs Predicted Body Mass")
plt.show()

If you have not already, you are encouraged to explore how changing hyperparameters, such as `max_features`, changes the structure of the tree. When doing so, consider how these changes affect the model's ability to balance the bias-variance tradeoff (underfitting versus overfitting).